# Google Play Store Analysis
## CRISP-DM + SEMMA + KDD Machine Learning Experiment

**Objective:** Predict whether a Google Play Store app has a high rating (`Rating >= 4.0`) using Random Forest Classification.

> Upload `googleplaystore.csv` when prompted. This notebook is designed to run directly in Google Colab.


In [4]:
# ============================================
# IMPORT LIBRARIES AND UPLOAD DATASET
# ============================================

import pandas as pd
import numpy as np

from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

uploaded = files.upload()

filename = list(uploaded.keys())[0]
print("Uploaded:", filename)

df = pd.read_csv(filename)

print("Dataset Shape:", df.shape)
df.head()


Saving googleplaystore.csv to googleplaystore (1).csv
Uploaded: googleplaystore (1).csv
Dataset Shape: (10841, 13)


,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


## CRISP-DM — Data Understanding

In [5]:
# ============================================
# CRISP-DM - DATA UNDERSTANDING
# ============================================

print("Shape:")
print(df.shape)

print("\nColumns:")
print(df.columns)

print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDataset Information:")
df.info()


Shape:
(10841, 13)

Columns:
Index(['App', 'Category', 'Rating', 'Reviews', 'Size', 'Installs', 'Type',
       'Price', 'Content Rating', 'Genres', 'Last Updated', 'Current Ver',
       'Android Ver'],
      dtype='object')

Data Types:
App                object
Category           object
Rating            float64
Reviews            object
Size               object
Installs           object
Type               object
Price              object
Content Rating     object
Genres             object
Last Updated       object
Current Ver        object
Android Ver        object
dtype: object

Missing Values:
App                  0
Category             0
Rating            1474
Reviews              0
Size                 0
Installs             0
Type                 1
Price                0
Content Rating       1
Genres               0
Last Updated         0
Current Ver          8
Android Ver          3
dtype: int64

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10841 entr

## Data Cleaning

In [6]:
# ============================================
# DATA CLEANING
# ============================================

df = df.drop_duplicates()

df["Reviews"] = pd.to_numeric(
    df["Reviews"], errors="coerce"
)

df["Installs"] = pd.to_numeric(
    df["Installs"]
    .astype(str)
    .str.replace("+", "", regex=False)
    .str.replace(",", "", regex=False),
    errors="coerce"
)

df["Price"] = pd.to_numeric(
    df["Price"]
    .astype(str)
    .str.replace("$", "", regex=False),
    errors="coerce"
)

df["Rating"] = pd.to_numeric(
    df["Rating"], errors="coerce"
)

# Keep only valid Google Play Store ratings
df = df[
    (df["Rating"] >= 1) &
    (df["Rating"] <= 5)
]

df = df.dropna(subset=["Rating"])

print("Cleaned Shape:", df.shape)
df.head()


Cleaned Shape: (8892, 13)


,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159.0,19M,10000.0,Free,0.0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967.0,14M,500000.0,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510.0,8.7M,5000000.0,Free,0.0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644.0,25M,50000000.0,Free,0.0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967.0,2.8M,100000.0,Free,0.0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


## Create Machine Learning Target

In [7]:
# ============================================
# CREATE TARGET VARIABLE
# ============================================

# 1 = High Rating (Rating >= 4.0)
# 0 = Lower Rating (Rating < 4.0)

df["High_Rating"] = np.where(
    df["Rating"] >= 4.0, 1, 0
)

print("Target Distribution:")
print(df["High_Rating"].value_counts())

print("\nTarget Percentage:")
print(
    df["High_Rating"]
    .value_counts(normalize=True)
    .mul(100)
)


Target Distribution:
High_Rating
1    6947
0    1945
Name: count, dtype: int64

Target Percentage:
High_Rating
1    78.126406
0    21.873594
Name: proportion, dtype: float64


## Feature Selection

`Rating` is **not** used as an input feature because it is used to create `High_Rating`. This prevents target leakage.


In [8]:
# ============================================
# FEATURE SELECTION
# ============================================

features = [
    "Reviews",
    "Installs",
    "Price",
    "Category",
    "Type",
    "Content Rating",
    "Genres"
]

categorical_columns = [
    "Category",
    "Type",
    "Content Rating",
    "Genres"
]

X = df[features].copy()
y = df["High_Rating"]

print("Features:")
print(X.columns)

print("\nTarget:")
print(y.name)


Features:
Index(['Reviews', 'Installs', 'Price', 'Category', 'Type', 'Content Rating',
       'Genres'],
      dtype='object')

Target:
High_Rating


## CRISP-DM — Data Preparation

In [9]:
# ============================================
# ENCODE CATEGORICAL VARIABLES
# ============================================

for column in categorical_columns:
    le = LabelEncoder()
    X[column] = le.fit_transform(X[column].astype(str))

imputer = SimpleImputer(strategy="median")
X = imputer.fit_transform(X)

print("Missing values handled.")
print("Feature shape:", X.shape)


Missing values handled.
Feature shape: (8892, 7)


In [10]:
# ============================================
# TRAIN TEST SPLIT
# ============================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)


Training data: (7113, 7)
Testing data: (1779, 7)


## CRISP-DM — Modeling

In [11]:
# ============================================
# CRISP-DM - MODELING
# ============================================

model_crisp = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model_crisp.fit(X_train, y_train)

print("CRISP-DM Random Forest Model Trained.")


CRISP-DM Random Forest Model Trained.


## CRISP-DM — Evaluation

In [12]:
# ============================================
# CRISP-DM - EVALUATION
# ============================================

prediction_crisp = model_crisp.predict(X_test)

accuracy_crisp = accuracy_score(
    y_test,
    prediction_crisp
)

print("CRISP-DM Accuracy:", accuracy_crisp * 100, "%")

print("\nClassification Report:")
print(classification_report(y_test, prediction_crisp))


CRISP-DM Accuracy: 74.81731309724564 %

Classification Report:
              precision    recall  f1-score   support

           0       0.41      0.33      0.37       389
           1       0.82      0.86      0.84      1390

    accuracy                           0.75      1779
   macro avg       0.61      0.60      0.60      1779
weighted avg       0.73      0.75      0.74      1779



## SEMMA — Sample

In [13]:
# ============================================
# SEMMA - SAMPLE
# ============================================

df_semma = df.copy()

print("SEMMA Sample Shape:")
print(df_semma.shape)

df_semma.head()


SEMMA Sample Shape:
(8892, 14)


,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,High_Rating
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159.0,19M,10000.0,Free,0.0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up,1
1,Coloring book moana,ART_AND_DESIGN,3.9,967.0,14M,500000.0,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,0
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510.0,8.7M,5000000.0,Free,0.0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up,1
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644.0,25M,50000000.0,Free,0.0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up,1
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967.0,2.8M,100000.0,Free,0.0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up,1


## SEMMA — Explore

In [14]:
# ============================================
# SEMMA - EXPLORE
# ============================================

print("Dataset Information:")
df_semma.info()

print("\nDescriptive Statistics:")
print(
    df_semma[
        ["Rating", "Reviews", "Installs", "Price"]
    ].describe()
)

print("\nHigh Rating Distribution:")
print(df_semma["High_Rating"].value_counts())


Dataset Information:
<class 'pandas.core.frame.DataFrame'>
Index: 8892 entries, 0 to 10840
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   App             8892 non-null   object 
 1   Category        8892 non-null   object 
 2   Rating          8892 non-null   float64
 3   Reviews         8892 non-null   float64
 4   Size            8892 non-null   object 
 5   Installs        8892 non-null   float64
 6   Type            8892 non-null   object 
 7   Price           8892 non-null   float64
 8   Content Rating  8892 non-null   object 
 9   Genres          8892 non-null   object 
 10  Last Updated    8892 non-null   object 
 11  Current Ver     8888 non-null   object 
 12  Android Ver     8890 non-null   object 
 13  High_Rating     8892 non-null   int64  
dtypes: float64(4), int64(1), object(9)
memory usage: 1.3+ MB

Descriptive Statistics:
            Rating       Reviews      Installs        Price
count  88

## SEMMA — Modify

In [15]:
# ============================================
# SEMMA - MODIFY
# ============================================

X_semma = df_semma[features].copy()
y_semma = df_semma["High_Rating"]

for column in categorical_columns:
    le = LabelEncoder()
    X_semma[column] = le.fit_transform(
        X_semma[column].astype(str)
    )

X_semma = SimpleImputer(
    strategy="median"
).fit_transform(X_semma)

print("SEMMA data modified.")
print("Feature shape:", X_semma.shape)


SEMMA data modified.
Feature shape: (8892, 7)


In [16]:
# ============================================
# SEMMA - TRAIN TEST SPLIT
# ============================================

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_semma,
    y_semma,
    test_size=0.20,
    random_state=42,
    stratify=y_semma
)


## SEMMA — Model

In [17]:
# ============================================
# SEMMA - MODEL
# ============================================

model_semma = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model_semma.fit(X_train_s, y_train_s)

print("SEMMA Random Forest Model Trained.")


SEMMA Random Forest Model Trained.


## SEMMA — Assess

In [18]:
# ============================================
# SEMMA - ASSESS
# ============================================

prediction_semma = model_semma.predict(X_test_s)

accuracy_semma = accuracy_score(
    y_test_s,
    prediction_semma
)

print("SEMMA Accuracy:", accuracy_semma * 100, "%")

print("\nClassification Report:")
print(classification_report(y_test_s, prediction_semma))


SEMMA Accuracy: 74.81731309724564 %

Classification Report:
              precision    recall  f1-score   support

           0       0.41      0.33      0.37       389
           1       0.82      0.86      0.84      1390

    accuracy                           0.75      1779
   macro avg       0.61      0.60      0.60      1779
weighted avg       0.73      0.75      0.74      1779



## KDD — Selection

In [19]:
# ============================================
# KDD - SELECTION
# ============================================

df_kdd = df.copy()

selected_columns = [
    "Reviews",
    "Installs",
    "Price",
    "Category",
    "Type",
    "Content Rating",
    "Genres",
    "High_Rating"
]

df_kdd = df_kdd[selected_columns]

print("Selected Columns:")
print(df_kdd.columns)

print("\nShape:")
print(df_kdd.shape)


Selected Columns:
Index(['Reviews', 'Installs', 'Price', 'Category', 'Type', 'Content Rating',
       'Genres', 'High_Rating'],
      dtype='object')

Shape:
(8892, 8)


## KDD — Preprocessing

In [20]:
# ============================================
# KDD - PREPROCESSING
# ============================================

X_kdd = df_kdd.drop("High_Rating", axis=1)
y_kdd = df_kdd["High_Rating"]

for column in categorical_columns:
    le = LabelEncoder()
    X_kdd[column] = le.fit_transform(
        X_kdd[column].astype(str)
    )

imputer_kdd = SimpleImputer(strategy="median")
X_kdd = imputer_kdd.fit_transform(X_kdd)

print("KDD preprocessing completed.")


KDD preprocessing completed.


## KDD — Transformation

In [21]:
# ============================================
# KDD - TRANSFORMATION
# ============================================

X_train_k, X_test_k, y_train_k, y_test_k = train_test_split(
    X_kdd,
    y_kdd,
    test_size=0.20,
    random_state=42,
    stratify=y_kdd
)

print("Training Shape:", X_train_k.shape)
print("Testing Shape:", X_test_k.shape)


Training Shape: (7113, 7)
Testing Shape: (1779, 7)


## KDD — Data Mining

In [22]:
# ============================================
# KDD - DATA MINING
# ============================================

model_kdd = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model_kdd.fit(X_train_k, y_train_k)

print("KDD Random Forest Model Trained.")


KDD Random Forest Model Trained.


## KDD — Interpretation

In [23]:
# ============================================
# KDD - INTERPRETATION
# ============================================

prediction_kdd = model_kdd.predict(X_test_k)

accuracy_kdd = accuracy_score(
    y_test_k,
    prediction_kdd
)

print("KDD Accuracy:", accuracy_kdd * 100, "%")

print("\nClassification Report:")
print(classification_report(y_test_k, prediction_kdd))


KDD Accuracy: 74.81731309724564 %

Classification Report:
              precision    recall  f1-score   support

           0       0.41      0.33      0.37       389
           1       0.82      0.86      0.84      1390

    accuracy                           0.75      1779
   macro avg       0.61      0.60      0.60      1779
weighted avg       0.73      0.75      0.74      1779



## Final Comparison

In [24]:
# ============================================
# FINAL COMPARISON
# ============================================

results = pd.DataFrame({
    "Methodology": [
        "CRISP-DM",
        "SEMMA",
        "KDD"
    ],
    "Accuracy": [
        accuracy_crisp * 100,
        accuracy_semma * 100,
        accuracy_kdd * 100
    ]
})

print(results)

print("\nSorted by Accuracy:")
display(
    results.sort_values(
        "Accuracy",
        ascending=False
    ).reset_index(drop=True)
)


  Methodology   Accuracy
0    CRISP-DM  74.817313
1       SEMMA  74.817313
2         KDD  74.817313

Sorted by Accuracy:


,Methodology,Accuracy
0,CRISP-DM,74.817313
1,SEMMA,74.817313
2,KDD,74.817313


## Conclusion

The three methodologies use the same Google Play Store classification problem and Random Forest model. The final accuracy comparison helps identify whether the preprocessing and workflow produce consistent results across CRISP-DM, SEMMA, and KDD.
